In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
import pandas as pd
from sklearn.model_selection import train_test_split as tts
from sklearn.feature_extraction.text import CountVectorizer
from time import time

In [2]:
# Get the document names
def get_target_vect(labels, emnedict):
    emneset = set(emnedict.keys())
    Y = np.zeros([len(labels), len(emneset)], dtype=int)
    for i, v in enumerate(labels):
        lab_set = set(v).intersection(emneset)
        for l in lab_set:
            Y[i, emnedict[l]] = 1
    return Y


documents = sorted([x for x in os.listdir('sammendrag')])
doc_names = [x.lower().removesuffix('.txt') for x in documents]
documents = [f"sammendrag/{x}" for x in documents]

# Get the labels
filename = "InnstillingerMedEmneord.csv"
df = pd.read_csv(filename, sep=';')
df['Filnavn'] = df['Filnavn'].apply(lambda x: x.lower())

labels = []
for doc_name in doc_names:
    labels.append([x.lower() for x in df[df['Filnavn'] == f"{doc_name}.xml"]['emneord'].values])

all_labels = []
for x in labels:
    all_labels += x


N_LABELS = 10
tags, counts = np.unique(all_labels, return_counts=True)
common_tags_ind = np.argsort(-counts)[:N_LABELS]
print("Most common labels\n", "-"*20)
    
emnedict_reduced = {}
for i, tag in enumerate(common_tags_ind):
    print(f"{tags[tag]} -- {counts[tag]}")
    emnedict_reduced[str(tags[tag])] = i
print(f"\nReduced emnedict: {emnedict_reduced}")

Y = get_target_vect(labels, emnedict_reduced)
non_empty_rows = np.arange(Y.shape[0])[Y.sum(axis=1) != 0]

# Split into train-test
train_ind, test_ind = tts(non_empty_rows, test_size=0.25, random_state=1)
train_doc = (np.array(documents)[train_ind]).tolist()
test_doc = np.array(documents)[test_ind]
Y_train = Y[train_ind].astype(np.uint32)
Y_test = Y[test_ind].astype(np.uint32)


# ========================
# Fit the count vectorizer
# ========================
vect = CountVectorizer(
    input='filename',
    ngram_range=(1, 2),
    max_features=30000,
    binary=True,
)

tic = time()
vect.fit(train_doc)
toc = time()
print(f"Vectorizer trained in {toc - tic:.3f} seconds")


# The Tsetlin machine is only trained for 12 epochs where the test-f1 dropped while train-f1 still increased.
with open(f"tm_model.pickle", 'rb') as infile:
    tm = pickle.load(infile)

L = tm.get_literals()
W = tm.get_weights()

# Calculate the literal difference importance
X_test = vect.transform(test_doc).toarray().astype(np.uint32)
reverse_emne = {v:k for k,v in emnedict_reduced.items()}

Most common labels
 --------------------
eu/eøs -- 669
trygder -- 456
helsevesen -- 391
innvandrere -- 328
politi og påtalemyndighet -- 300
traktater -- 273
forurensning -- 254
olje og gass -- 251
internasjonalt samarbeid -- 208
forskning -- 207

Reduced emnedict: {'eu/eøs': 0, 'trygder': 1, 'helsevesen': 2, 'innvandrere': 3, 'politi og påtalemyndighet': 4, 'traktater': 5, 'forurensning': 6, 'olje og gass': 7, 'internasjonalt samarbeid': 8, 'forskning': 9}
Vectorizer trained in 5.793 seconds


In [29]:
def get_local_importance(X, target, L, W):
    n_features = L.shape[-1]//2
    xt = X[target]
    filter = np.logical_and(xt == 1, W[target, :, target] >= 0)
    S = (L[target, filter, :] * W[target, filter, target].reshape(-1, 1)).sum(axis=0)
    pos_lit = S[:n_features]
    ind = np.argsort(-pos_lit)
    imp_score = pos_lit[ind]
    return imp_score, ind


sample_id = 78
threshold = 0.05

# Transform the sample document from text to active clauses via the tokenizer
x = X_test[sample_id].reshape(1, -1)
x_transformed = tm.transform(x.astype(np.uint32))
x_transformed = x_transformed.reshape(N_LABELS, -1)

# Print the predictions and true labels
Y_pred, class_sum = tm.predict(x)
predicted_targets = np.nonzero(Y_pred[0])[0]
print("Predicted", [reverse_emne[x] for x in predicted_targets])
print("True     ", [reverse_emne[x] for x in np.nonzero(Y_test[sample_id])[0]])
print('\n Targets =  ', list(emnedict_reduced.keys()))
print(f"Class sums = {class_sum}")

print()
for target in predicted_targets:
    impscore, ind = get_local_importance(x_transformed, target, L, W)
    important_words = vect.get_feature_names_out()[ind]
    impscore /= tm.T 
    score_filter = impscore > threshold
    print(important_words[score_filter])
    print(impscore[score_filter].round(3))
    print()

Predicted ['eu/eøs', 'trygder', 'helsevesen']
True      ['trygder', 'helsevesen']

 Targets =   ['eu/eøs', 'trygder', 'helsevesen', 'innvandrere', 'politi og påtalemyndighet', 'traktater', 'forurensning', 'olje og gass', 'internasjonalt samarbeid', 'forskning']
Class sums = [[   83.  2234.  3788. -1730. -2916. -4031. -1584. -1258. -1198.  -429.]]

[]
[]

['pensjon' 'arbeid' '70' 'statens' 'kroner' 'redusert']
[0.337 0.125 0.083 0.065 0.056 0.052]

['pasienter' 'helse og' 'helse' 'det fremmes' 'sykehus' 'forslag']
[0.408 0.351 0.307 0.226 0.075 0.073]



In [30]:
print(test_doc[sample_id], '\n')
with open(test_doc[sample_id], 'r') as f:
    print(f.read())

sammendrag/si057-02.txt 

Sosial- og helsedepartementet legger i proposisjonen fram
forslag om endringer av bevilgningene under enkelte kapitler på statsbudsjettet
for 2001. Det fremmes forslag om økning av utgiftene med
netto 946 mill. kroner. Forslagene under enkelte kapitler: Kap. 600 Sosial- og helsedepartementet Post 1 Driftsutgifter foreslås økt
med 7 mill. kroner. Kap. 604 Etat for rådssekretariater og enkelte
helse- og sosialfaglige oppgaver m.v. Post 1 Driftsutgifter foreslås økt
med 1 mill. kroner. Post 70 Tilskudd til frivillighetssentraler
foreslås redusert med 1,65 mill. kroner. Kap. 614 Utvikling av sosialtjenesten, tiltak for rusmiddelmisbrukere
m.v. Post 63 Utvikling av sosialtjenesten og rusmiddeltiltak
foreslås økt med 5 mill. kroner. Kap. 660 Krigspensjon Post 70 Tilskudd, militære foreslås
redusert med 9 mill. kroner. Post 71 Tilskudd, sivile foreslås redusert
med 27 mill. kroner. Kap. 666 Avtalefestet pensjon (AFP) Post 70 Tilskudd foreslås økt
med 14 mill. kroner.